In [2]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("1. Veriler yükleniyor ve birleştiriliyor...")
# HAM (Orijinal) dosyaları okuyoruz
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

test_ids = test_df["PassengerId"]
y_train = train_df["Transported"]

# Aynı işlemleri görmek için test ve train geçici olarak birleştirilir
train_X = train_df.drop("Transported", axis=1)
tum_veri = pd.concat([train_X, test_df], axis=0).reset_index(drop=True)


print("2. Özellik Mühendisliği (Feature Engineering) yapılıyor...")
# A. Toplam Harcama Sütunu Üretme (Sizi tuzağa düşüren ama çok puan getiren kısım)
harcama_sutunlari = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
tum_veri[harcama_sutunlari] = tum_veri[harcama_sutunlari].fillna(0)
tum_veri['Total_Spend'] = tum_veri[harcama_sutunlari].sum(axis=1)

# B. Kabin Parçalama (Güverte / Numara / Taraf) -> Bu skoru çok artırır!
tum_veri[['Cabin_Deck', 'Cabin_Num', 'Cabin_Side']] = tum_veri['Cabin'].str.split('/', expand=True)

# Gereksiz sütunları atalım
tum_veri = tum_veri.drop(["PassengerId", "Name", "Cabin"], axis=1)


print("3. Eksik Veriler Dolduruluyor...")
sayisal_sutunlar = tum_veri.select_dtypes(include=['float64', 'int64']).columns
for col in sayisal_sutunlar:
    tum_veri[col] = tum_veri[col].fillna(tum_veri[col].median()).infer_objects(copy=False)

kategorik_sutunlar = tum_veri.select_dtypes(include=['object', 'bool']).columns
for col in kategorik_sutunlar:
    tum_veri[col] = tum_veri[col].fillna(tum_veri[col].mode()[0]).infer_objects(copy=False)


print("4. Kategorik Sütunlar Sayısallaştırılıyor...")
tum_veri = pd.get_dummies(tum_veri, drop_first=True)


print("5. Veriler Tekrar Ayrılıyor ve Ölçekleniyor...")
# İlk 8693 satır train.csv'ye aittir
X_train = tum_veri.iloc[:len(train_df)]
X_test = tum_veri.iloc[len(train_df):]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


print("6. Şampiyon Model Eğitiliyor ve Tahmin Yapılıyor...")
# Daha önce bulduğumuz iyi ayarlarla
model = XGBClassifier(learning_rate=0.1, max_depth=5, n_estimators=100, random_state=42, eval_metric='logloss')
model.fit(X_train_scaled, y_train)

tahminler = model.predict(X_test_scaled)
tahminler_bool = [True if val == 1 else False for val in tahminler]


print("7. Dosya Kaydediliyor...")
sub = pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": tahminler_bool
})
sub.to_csv("submission_kesin_cozum.csv", index=False)

print("\nBAŞARILI! 🎉 'submission_kesin_cozum.csv' dosyası Kaggle'a yüklenmeye hazır!")

1. Veriler yükleniyor ve birleştiriliyor...
2. Özellik Mühendisliği (Feature Engineering) yapılıyor...
3. Eksik Veriler Dolduruluyor...
4. Kategorik Sütunlar Sayısallaştırılıyor...
5. Veriler Tekrar Ayrılıyor ve Ölçekleniyor...
6. Şampiyon Model Eğitiliyor ve Tahmin Yapılıyor...
7. Dosya Kaydediliyor...

BAŞARILI! 🎉 'submission_kesin_cozum.csv' dosyası Kaggle'a yüklenmeye hazır!
